# Hi-res 1601 — non-image feature zoo (modal / indicators / FRF) · GPU

The scientifically-sound complement to the CFDAC-image notebooks: the feature families that transferred **best** in the 128 baseline (modal, indicators) plus raw FRF, with **proper MLP / RandomForest / XGBoost / 1-D CNN / transformer**. Features are standardised on the synth-train fold and applied zero-shot to experiment; NN models train to convergence with checkpoint/resume; trees fit once. Set a GPU runtime; add a `GH_TOKEN` secret for autosave to `colab-hires-tabular`.

## 1 · Bootstrap

In [ ]:
import os, sys, subprocess
GH_USER='grcarmenaty'; WORK='/content'; os.chdir(WORK)
def _tok():
    try:
        from google.colab import userdata; return userdata.get('GH_TOKEN')
    except Exception: return os.environ.get('GH_TOKEN')
def clone(repo, branch, dst):
    if os.path.isdir(dst): print('exists', dst); return
    t=_tok(); auth=f'{t}@' if t else ''
    url=f'https://{auth}github.com/{GH_USER}/{repo}.git'
    assert subprocess.run(['git','clone','--depth','1','-b',branch,url,dst]).returncode==0, \
        f'clone failed {repo}@{branch} (private? add a GH_TOKEN Colab secret)'
clone('phd_lanl','main','/content/PhD_LANL')
clone('pymodal','master','/content/pymodal')   # sibling dir the scripts expect
for p in ('/content/PhD_LANL','/content/pymodal'):
    if p not in sys.path: sys.path.insert(0,p)
os.chdir('/content/PhD_LANL')
# Harden git's HTTP transport against Drive-mounted-Colab flakiness (the 408s):
for _k,_v in [('http.postBuffer','524288000'),('http.version','HTTP/1.1'),
              ('http.lowSpeedLimit','1000'),('http.lowSpeedTime','300')]:
    subprocess.run(['git','config','--global',_k,_v])
subprocess.run([sys.executable,'-m','pip','-q','install','timm','h5py','scikit-learn','pint','pyFRF','audiomentations'])
import torch
print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),'|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - set a GPU runtime!')

## 2 · Regenerate 1601-bin features

In [ ]:
import subprocess, sys, os, glob, json, h5py, numpy as np
from pathlib import Path
REPO=Path(os.getcwd())
def run(cmd): print('>>',' '.join(cmd)); assert subprocess.run(cmd).returncode==0, cmd
if not (REPO/'dataset'/'features_hires.h5').exists():
    run([sys.executable,'ml_pipeline/generate_dataset.py','--out','dataset_hires','--n-t','4096','--fs','256'])
    run([sys.executable,'ml_pipeline/build_hires_synth_features.py'])
if not (REPO/'experimental_frfs.h5').exists():
    with open('experimental_frfs.h5','wb') as o:
        for p in sorted(glob.glob('experimental_frfs_chunks/experimental_frfs.h5.part_*')):
            o.write(open(p,'rb').read())
if not (REPO/'dataset'/'experimental_features.h5').exists():
    from ml_pipeline.evaluate import primary_op
    with h5py.File('experimental_frfs.h5','r') as f: names=json.loads(f.attrs['case_names_json'])
    n=len(names); tc=np.zeros(n,np.int8); st=np.full(n,-1,np.int8); en=np.full(n,-1,np.int8); sv=np.zeros(n,np.float32)
    for i,nm in enumerate(names):
        op=primary_op(nm); tc[i]=op['type_code']; st[i]=op['storey']; en[i]=op['end']; sv[i]=op['severity']
    dt=h5py.string_dtype('utf-8')
    with h5py.File('dataset/experimental_features.h5','w') as o:
        o.create_dataset('names',data=np.array(names,dtype=object),dtype=dt)
        o.create_dataset('type_code',data=tc); o.create_dataset('storey',data=st)
        o.create_dataset('end',data=en); o.create_dataset('severity',data=sv)
if not (REPO/'dataset'/'experimental_features_hires.h5').exists():
    run([sys.executable,'ml_pipeline/build_hires_exp_features.py'])
print('features ready')

## 3 · Config + precompute feature caches (once, on Drive)

In [ ]:
import torch, numpy as np, h5py
from pathlib import Path
from ml_pipeline import hires_tab as T
from ml_pipeline.tasks import build_targets
from ml_pipeline.train import make_split
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ===================== CONFIG (edit me) =====================
MODELS = ['mlp','rf','xgb','cnn1d','transformer1d']
TASKS  = ['binary','col_location','mass_location','severity','type',
          'is_bolt','is_crack','is_mass','is_hole','is_pristine']
SUBSAMPLE = 4000          # synth samples per task (raise toward 10000 for more data)
BATCH     = 256           # tabular/seq NN batch (tiny models -> large batch fine)
AUTOSAVE_GITHUB   = True
FAMILY            = 'tabular'
GH_RESULTS_BRANCH = 'colab-hires-tabular'
# feature/model compatibility lives in T.TAB_MODEL_FEATURES:
#   mlp: modal,indicators,frf_mag,frf_realimag,timeseries | rf,xgb: modal,indicators
#   cnn1d,transformer1d: frf_mag,frf_realimag,timeseries
# NB timeseries is reconstructed from the FRF (IFFT*chirp) identically for synth &
# exp — the experimental set has no measured timeseries. To run ONE cell:
#   CELLS = [('is_hole','cnn1d','timeseries')]
CELLS = T.tab_cells(MODELS, TASKS)
print(len(CELLS),'cells queued across', MODELS)
# ===========================================================

try:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/hires_cfdac/tabular')
except Exception:
    OUT = Path('results_hires_zoo_tabular')
OUT.mkdir(parents=True, exist_ok=True); (OUT/'cache').mkdir(exist_ok=True)

# Pick up cells already trained in a past run (seed from the results branch).
import subprocess as _sp, os as _os
try:
    _sp.run(['git','-C','/content/PhD_LANL','fetch','--depth','1','origin',GH_RESULTS_BRANCH], capture_output=True)
    _ls=_sp.run(['git','-C','/content/PhD_LANL','ls-tree','-r','--name-only','origin/'+GH_RESULTS_BRANCH],capture_output=True,text=True).stdout
    (OUT/'per_case').mkdir(parents=True, exist_ok=True); _n=0
    for _l in _ls.splitlines():
        if 'results_hires_zoo/'+FAMILY+'/per_case/' in _l and _l.endswith('.json'):
            _name=_os.path.basename(_l)
            if not (OUT/'per_case'/_name).exists():
                _b=_sp.run(['git','-C','/content/PhD_LANL','show','origin/'+GH_RESULTS_BRANCH+':'+_l],capture_output=True,text=True).stdout
                if _b: (OUT/'per_case'/_name).write_text(_b); _n+=1
    print('picked up',_n,'already-trained cells from',GH_RESULTS_BRANCH)
except Exception as _e: print('branch pickup skipped:', _e)

SYN = 'dataset/features_hires.h5'; EXP = 'dataset/experimental_features_hires.h5'
with h5py.File(SYN,'r') as f:
    syn_tasks = build_targets(f['type_code'][:].astype('int64'),f['storey'][:].astype('int64'),
                              f['end'][:].astype('int64'),f['severity'][:].astype('float32'))
with h5py.File(EXP,'r') as f:
    exp_tasks = build_targets(f['type_code'][:].astype('int64'),f['storey'][:].astype('int64'),
                              f['end'][:].astype('int64'),f['severity'][:].astype('float32'))
    exp_names = [str(s) for s in f['names'][:]]

# Precompute each feature ONCE for ALL samples (cached on Drive), so indicators
# (a 1601 CFDAC per sample) are never recomputed per task.
feats_used = sorted({f for (_,_,f) in CELLS})
CACHE = {}
for ft in feats_used:
    print('building cache:', ft, '(indicators = slow: 1601 CFDAC/sample)')
    Xs = T.build_feature_cache(SYN, ft, OUT/'cache'/f'{ft}_syn.npy')
    Xe = T.build_feature_cache(EXP, ft, OUT/'cache'/f'{ft}_exp.npy')
    CACHE[ft] = (Xs, Xe)
print('caches:', {k:(tuple(v[0].shape),tuple(v[1].shape)) for k,v in CACHE.items()})
print('device', DEV, '| amp', T._amp_dtype(DEV))

## 4 · Run the grid (skip-if-exists; NN resume from checkpoint)

In [ ]:
import torch, os, shutil, subprocess
def _tok():
    try:
        from google.colab import userdata; return userdata.get('GH_TOKEN')
    except Exception: return os.environ.get('GH_TOKEN')
GH_TOKEN = _tok()
if AUTOSAVE_GITHUB and not GH_TOKEN:
    print('AUTOSAVE on but no GH_TOKEN -> Drive/zip only')

def git_autosave(msg):
    if not (AUTOSAVE_GITHUB and GH_TOKEN): return
    repo='/content/PhD_LANL'; dst=os.path.join(repo,'results_hires_zoo',FAMILY)
    os.makedirs(os.path.join(dst,'per_case'), exist_ok=True)
    if not getattr(git_autosave,'_merged',False):   # one-time: pull remote cells so a force-push never overwrites a fuller branch
        subprocess.run(['git','-C',repo,'fetch','--depth','1','origin',GH_RESULTS_BRANCH],capture_output=True)
        _rl=subprocess.run(['git','-C',repo,'ls-tree','-r','--name-only','origin/'+GH_RESULTS_BRANCH],capture_output=True,text=True).stdout
        for _l in _rl.splitlines():
            if '/per_case/' in _l and _l.endswith('.json'):
                _fp=os.path.join(str(OUT),'per_case',os.path.basename(_l))
                if not os.path.exists(_fp):
                    _bb=subprocess.run(['git','-C',repo,'show','origin/'+GH_RESULTS_BRANCH+':'+_l],capture_output=True,text=True).stdout
                    if _bb: open(_fp,'w').write(_bb)
        git_autosave._merged=True
    for fn in (os.listdir(os.path.join(OUT,'per_case')) if os.path.isdir(os.path.join(OUT,'per_case')) else []):
        if fn.endswith('.json'): shutil.copy(os.path.join(OUT,'per_case',fn), os.path.join(dst,'per_case',fn))
    if os.path.exists(os.path.join(OUT,'synth_test_tab.json')):
        shutil.copy(os.path.join(OUT,'synth_test_tab.json'), os.path.join(dst,'synth_test_tab.json'))
    cwd=os.getcwd(); os.chdir(repo)
    subprocess.run(['git','config','user.email','colab@gpu.run'])
    subprocess.run(['git','config','user.name','colab-gpu'])
    subprocess.run(['git','add','-f',f'results_hires_zoo/{FAMILY}/per_case',f'results_hires_zoo/{FAMILY}/synth_test_tab.json'])
    if subprocess.run(['git','diff','--cached','--quiet']).returncode!=0:
        subprocess.run(['git','commit','-q','-m',msg])
        url=f'https://{GH_TOKEN}@github.com/grcarmenaty/phd_lanl.git'
        import time as _t; ok=False
        for _a in range(5):
            r=subprocess.run(['git','push','--force',url,f'HEAD:{GH_RESULTS_BRANCH}'],capture_output=True,text=True)
            if r.returncode==0: ok=True; break
            _t.sleep(4*(2**_a))
        print('  autosave:', f'pushed -> {GH_RESULTS_BRANCH}' if ok else 'push failed after retries (Drive has results): '+r.stderr[-140:])
    os.chdir(cwd)

for (task, model, feature) in CELLS:
    try:
        T.run_tab_cell(task, model, feature, out_dir=OUT, dev=DEV, syn_tasks=syn_tasks,
                       exp_tasks=exp_tasks, Xsyn=CACHE[feature][0], Xexp=CACHE[feature][1],
                       exp_names=exp_names, make_split=make_split, subsample=SUBSAMPLE, batch=BATCH)
        git_autosave(f'colab autosave [tabular]: {task}/{model}/{feature}')
    except Exception as e:
        print('CELL FAILED', task, model, feature, '::', repr(e)[:200])
        if torch.cuda.is_available(): torch.cuda.empty_cache()
print('\nqueue done')

## 5 · Honest summary + zip download

In [ ]:
import json, numpy as np
from pathlib import Path
from collections import Counter
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
print(f"{'cell':<46}{'kind':>5}{'synth':>8}{'expMF1/R2':>11}{'expBal':>8}{'collapse':>9}")
print('-'*86)
for p in sorted((OUT/'per_case').glob('*_hires1601.json')):
    d=json.loads(p.read_text()); m=d['meta']; r=d['rows']
    yt=np.array([x['y_true'] for x in r]); yp=np.array([x['y_pred'] for x in r])
    name=f"{m['task']}/{m['model']}/{m['feature']}"
    if m['kind']=='cls':
        n=m['n_out']; bal=balanced_accuracy_score(yt,yp)
        mf1=f1_score(yt,yp,labels=list(range(n)),average='macro',zero_division=0)
        coll=(len(set(yp.tolist()))<=1) or (bal<=1/n+0.02)
        print(f"{name:<46}{'cls':>5}{(m.get('synth_test_macro_f1') or 0):>8.3f}{mf1:>11.3f}{bal:>8.3f}{str(coll):>9}")
    else:
        yt=yt.astype(float); yp=yp.astype(float); ss=np.sum((yt-yp)**2); st=np.sum((yt-yt.mean())**2)
        r2=1-ss/st if st>0 else 0
        print(f"{name:<46}{'reg':>5}{(m.get('synth_test_metric') or 0):>8.3f}{r2:>11.3f}{'-':>8}{'-':>9}")
import shutil
shutil.make_archive('/content/results_tabular','zip',str(OUT))
try:
    from google.colab import files; files.download('/content/results_tabular.zip')
except Exception as e: print('zip at /content/results_tabular.zip', e)

## 6 · (Optional) push results to the repo

In [ ]:
# Optional: force the full JSON snapshot to the results branch now (JSON only,
# no model weights). Same robust path as the per-cell autosave; safe to re-run.
import os, subprocess, shutil, glob, time as _t
tok=None
try:
    from google.colab import userdata; tok=userdata.get('GH_TOKEN')
except Exception: tok=os.environ.get('GH_TOKEN')
if not tok:
    print('No GH_TOKEN - download the zip from the cell above and hand it to the agent.')
else:
    repo='/content/PhD_LANL'; dst=os.path.join(repo,'results_hires_zoo',FAMILY)
    os.makedirs(os.path.join(dst,'per_case'), exist_ok=True)
    for fn in (os.listdir(os.path.join(OUT,'per_case')) if os.path.isdir(os.path.join(OUT,'per_case')) else []):
        if fn.endswith('.json'): shutil.copy(os.path.join(OUT,'per_case',fn), os.path.join(dst,'per_case',fn))
    for sj in glob.glob(os.path.join(OUT,'synth_test_*.json')):
        shutil.copy(sj, os.path.join(dst, os.path.basename(sj)))
    os.chdir(repo)
    subprocess.run(['git','config','user.email','colab@gpu.run']); subprocess.run(['git','config','user.name','colab-gpu'])
    subprocess.run(['git','add','-f',f'results_hires_zoo/{FAMILY}'])
    subprocess.run(['git','commit','-q','-m',f'hires {FAMILY} (GPU): manual JSON snapshot'])
    url=f'https://{tok}@github.com/grcarmenaty/phd_lanl.git'; ok=False
    for _a in range(5):
        r=subprocess.run(['git','push','--force',url,f'HEAD:{GH_RESULTS_BRANCH}'],capture_output=True,text=True)
        if r.returncode==0: ok=True; break
        _t.sleep(4*(2**_a))
    print(f'pushed -> {GH_RESULTS_BRANCH}' if ok else 'push failed after retries: '+r.stderr[-200:])